In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime
import urllib3

# SSL 경고 메시지를 끄기 위한 설정
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def crawl_pascucci():
    all_stores = []
    base_url = "https://www.pascucci.co.kr/store/storeList.asp?page="

    print("--- 크롤링을 시작합니다 (SSL 검증 우회) ---")

    for page in range(1, 49):
        url = f"{base_url}{page}"
        
        try:
            # verify=False를 추가하여 인증서 에러를 무시합니다.
            response = requests.get(url, verify=False)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')

            # 파스쿠찌 사이트의 실제 구조에 맞춰 tbody 내의 tr들을 가져옵니다.
            rows = soup.select('table tbody tr')

            # 만약 데이터가 없는 빈 페이지라면 중단하거나 건너뜁니다.
            if not rows:
                continue

            for row in rows:
                # 1. 매점명 (storeName)
                name_tag = row.find(class_='storeName')
                store_name = name_tag.get_text(strip=True) if name_tag else ""

                # 2. 지역 (storeArea)
                area_tag = row.find(class_='storeArea')
                store_area = area_tag.get_text(strip=True) if area_tag else ""

                # 3. 주소 (subject 클래스 안의 addr 클래스)
                subject_tag = row.find(class_='subject')
                addr_tag = subject_tag.find(class_='addr') if subject_tag else None
                address = addr_tag.get_text(strip=True) if addr_tag else ""

                # 빈 행이 수집되는 것을 방지 (데이터가 있을 때만 추가)
                if store_name:
                    all_stores.append({
                        "매점명": store_name,
                        "지역": store_area,
                        "주소": address
                    })

            current_time = datetime.now().strftime('%H:%M:%S')
            print(f"[{current_time}] {page} 페이지 완료 (누적 {len(all_stores)}개)")
            
            time.sleep(0.3)

        except Exception as e:
            print(f"Error at page {page}: {e}")

    if all_stores:
        df = pd.DataFrame(all_stores)
        df.to_csv("pascucci.csv", index=False, encoding="utf-8-sig")
        print(f"--- 총 {len(all_stores)}개의 매장 정보 저장 완료 (pascucci.csv) ---")
    else:
        print("수집된 데이터가 없습니다. 클래스명이나 페이지 구조를 다시 확인해야 합니다.")

if __name__ == "__main__":
    crawl_pascucci()

--- 크롤링을 시작합니다 (SSL 검증 우회) ---
[11:12:53] 1 페이지 완료 (누적 10개)
[11:12:53] 2 페이지 완료 (누적 20개)
[11:12:54] 3 페이지 완료 (누적 30개)
[11:12:55] 4 페이지 완료 (누적 40개)
[11:12:55] 5 페이지 완료 (누적 50개)
[11:12:56] 6 페이지 완료 (누적 60개)
[11:12:57] 7 페이지 완료 (누적 70개)
[11:12:57] 8 페이지 완료 (누적 80개)
[11:12:58] 9 페이지 완료 (누적 90개)
[11:12:59] 10 페이지 완료 (누적 100개)
[11:12:59] 11 페이지 완료 (누적 110개)
[11:13:00] 12 페이지 완료 (누적 120개)
[11:13:01] 13 페이지 완료 (누적 130개)
[11:13:01] 14 페이지 완료 (누적 140개)
[11:13:02] 15 페이지 완료 (누적 150개)
[11:13:03] 16 페이지 완료 (누적 160개)
[11:13:03] 17 페이지 완료 (누적 170개)
[11:13:04] 18 페이지 완료 (누적 180개)
[11:13:04] 19 페이지 완료 (누적 190개)
[11:13:05] 20 페이지 완료 (누적 200개)
[11:13:06] 21 페이지 완료 (누적 210개)
[11:13:06] 22 페이지 완료 (누적 220개)
[11:13:07] 23 페이지 완료 (누적 230개)
[11:13:08] 24 페이지 완료 (누적 240개)
[11:13:08] 25 페이지 완료 (누적 250개)
[11:13:09] 26 페이지 완료 (누적 260개)
[11:13:10] 27 페이지 완료 (누적 270개)
[11:13:10] 28 페이지 완료 (누적 280개)
[11:13:11] 29 페이지 완료 (누적 290개)
[11:13:12] 30 페이지 완료 (누적 300개)
[11:13:12] 31 페이지 완료 (누적 310개)
[11:13:13] 32 페이지 완료 (누적 3

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

# 1. CSV 파일 로드
file_path = 'pascucci.csv'
df = pd.read_csv(file_path)

# 2. MySQL 연결 설정 (사용자 정보에 맞게 수정 필요)
user = 'root'      # MySQL 사용자명 (예: root)
password = '1234'  # MySQL 비밀번호
host = 'localhost'          # 호스트 (예: 127.0.0.1)
port = '3306'               # 포트 번호
db_name = 'coffee_store'    # 대상 데이터베이스

# 3. 데이터베이스 생성 (이미 존재하면 무시)
# 먼저 MySQL 서버 자체에 연결하여 DB가 없으면 생성합니다.
temp_engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}")
with temp_engine.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {db_name}"))
    print(f"데이터베이스 '{db_name}'가 준비되었습니다.")

# 4. SQLAlchemy 엔진 생성 (coffee_store DB 연결)
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{db_name}?charset=utf8mb4")

# 5. 데이터프레임을 MySQL 테이블로 저장
# - name: 생성할 테이블 이름 ('the_venti')
# - if_exists: 'fail' (이미 있으면 오류), 'replace' (기존 삭제 후 생성), 'append' (기존 테이블에 추가)
# - index: False (데이터프레임의 인덱스는 컬럼으로 넣지 않음)
try:
    df.to_sql(name='pascucci', con=engine, if_exists='replace', index=False)
    print(f"테이블 'pascucci'에 총 {len(df)}건의 데이터가 성공적으로 저장되었습니다.")
except Exception as e:
    print(f"데이터 저장 중 오류 발생: {e}")

# 연결 종료
engine.dispose()

데이터베이스 'coffee_store'가 준비되었습니다.
테이블 'pascucci'에 총 477건의 데이터가 성공적으로 저장되었습니다.
